# 鶯歌工商 Mini AI Assistant & YKVS demo Colab 精簡版 v2

此 Notebook 保留 NCHU workshop 的 **Express → MCP Client → Python FastMCP → `search_school_info`** 架構，並沿用既有 `get_weather` MCP。分機更新、場館 79 筆、職員姓名/職稱/單位、班級分機、生輔組 4 份規定、法規 PDF→JSON 轉換器、獨立查詢頁，以及核對過的 115-1 行事曆，都已整合在 Notebook 內。

**資料不必逐項上傳。**只需把這一個 `.ipynb` 上傳/開啟於 Colab，照順序執行各格；資料包已內嵌。第一次仍需 Colab 下載 workshop 原始程式並安裝依賴。請先在 Colab Secrets 建立 `GEMINI_API_KEY`，僅供一般生成式問答使用；星期與日期為本機計算，校園資料走 MCP 快查。

`english_center.json` 已含 21 週（2026-08-30 至 2027-01-23）。Notebook 會檢查週數、79 筆場館、240 筆人員紀錄、57 班與 4 份生輔規定，不再要求另外上傳行事曆 JSON。

## 1｜取得 workshop 並安裝依賴
用途：從原 workshop 專案建立既有 Node/Express 與 Python MCP 環境；保留原專案，不另建查詢伺服器。第一次執行會下載程式及套件。

In [1]:
from pathlib import Path
import os, shutil, subprocess, time

REPO = Path('/content/nchu-mcp-workshop-2026')
PROJECT = REPO / 'mini-project'
if not PROJECT.exists():
    subprocess.run(['git', 'clone', '-q', 'https://github.com/UDICatNCHU/nchu-mcp-workshop-2026', str(REPO)], check=True)

if shutil.which('node') is None:
    subprocess.run('curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash - >/dev/null 2>&1 && sudo apt-get install -y nodejs >/dev/null 2>&1', shell=True, check=True)
if shutil.which('uv') is None:
    subprocess.run('curl -LsSf https://astral.sh/uv/install.sh | sh', shell=True, check=True)
    os.environ['PATH'] = str(Path.home() / '.local/bin') + ':' + os.environ['PATH']

subprocess.run(['npm', 'install', '--silent'], cwd=PROJECT / 'backend-node', check=True)
subprocess.run(['uv', 'sync', '--quiet'], cwd=PROJECT / 'mcp-server-py', check=True)
print('workshop 專案及依賴準備完成：', PROJECT)

workshop 專案及依賴準備完成： /content/nchu-mcp-workshop-2026/mini-project


## 2｜安裝內嵌的 YKVS 一體化更新資料
用途：自動套用查詢加速、分機與場館資料、生輔組 4 份規定、PDF→JSON 工具、獨立查詢頁與核對版 115-1 行事曆。只備份並覆蓋 YKVS 相關檔案，不修改 `config.json`、`web/index.html` 或其他 MCP 工具。

In [2]:
from pathlib import Path
from datetime import datetime
import base64, io, json, shutil, zipfile

PROJECT = Path('/content/nchu-mcp-workshop-2026/mini-project')
# 內嵌的一體化資料包，無需再從電腦選取多個 ZIP/JSON。
PACK_B64 = """..."""
archive = zipfile.ZipFile(io.BytesIO(base64.b64decode(PACK_B64)))
allowed = {
    'backend-node/server.js', 'backend-node/llm-client-openai.js', 'backend-node/quick-reply.js',
    'mcp-server-py/hello_tool.py', 'mcp-server-py/data/english_center.json', 'web/ykvs.html',
}
allowed_prefixes = ('mcp-server-py/data/ykvs_mvp_source_pack/', 'scripts/')
entries = [n for n in archive.namelist() if n in allowed or n.startswith(allowed_prefixes)]
if not entries:
    raise RuntimeError('內嵌資料包沒有可安裝檔案')
backup = Path('/content/ykvs_backup') / datetime.now().strftime('%Y%m%d_%H%M%S')
for rel in entries:
    target = PROJECT / rel
    if target.is_file():
        dest = backup / rel
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(target, dest)
for rel in entries:
    target = (PROJECT / rel).resolve()
    if PROJECT.resolve() not in target.parents:
        raise ValueError(f'拒絕寫入專案目錄之外：{rel}')
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(archive.read(rel))
print(f'YKVS 一體化資料安裝完成：{len(entries)} 個檔案')
print('備份位置：', backup)
print('保留原 workshop config.json、web/index.html 與其他 MCP tools。')

YKVS 一體化資料安裝完成：18 個檔案
備份位置： /content/ykvs_backup/20260924_054102
保留原 workshop config.json、web/index.html 與其他 MCP tools。


## 3｜檢查資料與 MCP 設定
用途：驗證 79 筆場館、至少 240 筆人員紀錄、57 班、4 份法規、21 週行事曆，以及原 workshop 的 `hello_tool`、`weather_tool` 註冊。

In [3]:
import json
from pathlib import Path

PROJECT = Path('/content/nchu-mcp-workshop-2026/mini-project')
DATA = PROJECT / 'mcp-server-py/data'
PACK = DATA / 'ykvs_mvp_source_pack/data'
staff = json.loads((PACK / 'staff_extensions.json').read_text(encoding='utf-8'))
classes = json.loads((PACK / 'class_extensions.json').read_text(encoding='utf-8'))
calendar = json.loads((DATA / 'english_center.json').read_text(encoding='utf-8'))
laws = sorted((PACK / 'regulations').glob('*.md'))
counts = {
    'staff': len(staff.get('分機紀錄', [])),
    'venues': len(staff.get('場館分機', [])),
    'classes': len(classes.get('班級分機', {})),
    'broadcast': len(classes.get('年級廣播', [])),
    'regulations': len(laws),
    'calendar_weeks': len(calendar.get('calendar_weeks', [])),
}
print('資料核對：', counts)
assert counts['staff'] >= 240, '人員資料筆數不足'
assert counts['venues'] == 79, '場館分機應為 79 筆'
assert counts['classes'] == 57, '班級分機應為 57 班'
assert counts['regulations'] == 4, '生輔組規定應為 4 份'
assert counts['calendar_weeks'] == 21, '115-1 行事曆應有 21 週'
assert (PROJECT / 'web/ykvs.html').exists()
assert (PROJECT / 'scripts/pdf_to_regulations_json.py').exists()
config = json.loads((PROJECT / 'config.json').read_text(encoding='utf-8'))
assert 'hello_tool' in config.get('mcpServers', {}) and 'weather_tool' in config.get('mcpServers', {})
print('檢查通過：MCP 工具設定、查詢頁、PDF→JSON 工具及行事曆都已備妥。')

資料核對： {'staff': 240, 'venues': 79, 'classes': 57, 'broadcast': 3, 'regulations': 4, 'calendar_weeks': 21}
檢查通過：MCP 工具設定、查詢頁、PDF→JSON 工具及行事曆都已備妥。


## 4｜讀取 Gemini Secret
用途：讓一般自由問答使用原 Gemini + MCP 流程。分機／法規／行事曆走 MCP 快查；星期日期本機計算；天氣沿用 `get_weather` MCP。金鑰只從 Colab Secrets 讀取，不印出、不寫入檔案。

In [20]:
import os
from google.colab import userdata

key = userdata.get('GEMINI_API_KEY')
if not key:
    raise ValueError('請先在 Colab 左側「秘密金鑰」建立 GEMINI_API_KEY，並允許此 Notebook 存取。')
os.environ['LLM_PROVIDER'] = 'openai'
os.environ['OPENAI_BASE_URL'] = 'https://generativelanguage.googleapis.com/v1beta/openai/'
os.environ['OPENAI_API_KEY'] = key
os.environ['OPENAI_MODEL'] = 'gemini-3.1-flash-lite'
del key
print('模型金鑰已讀取，內容不會顯示。')

模型金鑰已讀取，內容不會顯示。


## 5｜啟動原有 Express 與 MCP 服務
用途：重新啟動 workshop 既有 server.js，確認服務就緒；此步驟會連接設定檔中的 MCP servers。

In [15]:
import os, subprocess, time, urllib.request
from pathlib import Path

PROJECT = Path('/content/nchu-mcp-workshop-2026/mini-project')
subprocess.run(['bash', '-lc', "pkill -f '[n]ode server.js' || true"], capture_output=True)
log_path = Path('/content/server.log')
log_file = log_path.open('w')
server_proc = subprocess.Popen(['node', 'server.js'], cwd=PROJECT / 'backend-node', env=os.environ.copy(), stdout=log_file, stderr=subprocess.STDOUT, start_new_session=True)
for _ in range(40):
    time.sleep(1)
    try:
        with urllib.request.urlopen('http://127.0.0.1:3000', timeout=2) as r:
            if r.status == 200: break
    except Exception: pass
else:
    print(log_path.read_text(encoding='utf-8', errors='replace')[-5000:])
    raise RuntimeError('server 未在 40 秒內啟動；請查看上方錯誤記錄')
log = log_path.read_text(encoding='utf-8', errors='replace')
print('Express 啟動成功；MCP 工具記錄：')
print('\n'.join(line for line in log.splitlines() if 'search_school_info' in line or 'get_weather' in line or '[MCP]' in line) or '請確認 server log 中 hello_tool 與 weather_tool 已連線')

Express 啟動成功；MCP 工具記錄：
✓ hello_tool → search_school_info
✓ weather_tool → get_weather


## 6｜開啟獨立查詢頁並人工驗收
用途：建立 Colab proxy 新分頁連結；送出以下題目確認答案與來源。

In [16]:
from google.colab.output import eval_js
from IPython.display import display, HTML

base = eval_js('google.colab.kernel.proxyPort(3000)')
url = base.rstrip('/') + '/ykvs.html'
display(HTML(f'<a href="{url}" target="_blank" rel="noopener">開啟鶯歌工商 Mini AI Assistant & YKVS demo</a>'))
print("新分頁開啟後可測：生輔組分機、教師姓名、場館分機、請假規定、行事曆日期。")

新分頁開啟後可測：生輔組分機、教師姓名、場館分機、請假規定、行事曆日期。


### 建議驗收題
1. 鶯歌工商生輔組分機有哪些？（841、611）
2. 請查教師姓名的單位、職稱和分機。
3. 教學大樓 6D 電腦教室分機是多少？
4. 學生在家因病不能到校，返校後幾天內要補辦請假？請附學生請假規則來源。
5. 115-1 行事曆 2026/10/6 有哪些行程？
6. 今天星期幾？（應由台灣時區本機快速計算）

若 Gemini 回覆遇到 503 或逾時，先用分機、法規、行事曆快查題驗收 MCP 路徑；一般生成式問答仍會受模型服務狀態與免費額度影響。